# Embedding Experiments — Stratified Permutation Tests

**NOTE**: If you see an `AttributeError` after an update, please **Restart the Kernel** to ensure the new function definitions are loaded.

Four analyses using `stratified_permutation_test` from `orgpackage/tester.py`:

| # | Analysis | Factor | Stratification |
|---|----------|--------|----------------|
| 1 | **Shot count** | 0 vs 1 vs Few-shot | Model |
| 2 | **Model similarity** | Model comparison | Domain |
| 3 | **Classifier head** | LR vs SVM | Model |
| 4 | **Model classifier** | Model comparison | Domain |

Results cached in `results/correctness_tables/`. Global **Holm correction** applied.

## 0. Imports & Setup

In [ ]:
import os, sys, ast, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity
from statsmodels.stats.multitest import multipletests

project_root = os.path.abspath(".")
if project_root not in sys.path: sys.path.insert(0, project_root)
os.chdir(project_root)

from orgpackage.aux import load_experiments, load_trained_model, load_dataset
from orgpackage.config import DOMAIN_CLASSES_CORR
from orgpackage.tester import stratified_permutation_test

warnings.filterwarnings("ignore")
CORRECTNESS_DIR = "./results/correctness_tables"
os.makedirs(CORRECTNESS_DIR, exist_ok=True)
DOMAINS = ["medical", "administrative", "education"]
ALL_LABELS = ["hospital", "university_hospital", "local_government", 
              "primary_school", "secondary_school"]
print("Setup complete.")

## 1. Load Data

In [ ]:
exps = load_experiments()
emb_exps = exps[exps["Technique"] == "embedding"]
sim_exps = emb_exps[emb_exps["Method"] == "similarity"]
clf_exps = emb_exps[emb_exps["Method"] == "classifier"]

print("Loading master dataset…")
dataset = load_dataset()
dataset["instance"] = dataset["instance"].str.strip().str.lower().str.replace('https://', 'http://')

# Ensure ALL_LABELS are present in dataset
for c in ALL_LABELS: 
    if c not in dataset.columns: dataset[c] = 0
    else: dataset[c] = dataset[c].fillna(0).astype(int)
    
print(f"{len(dataset):,} items loaded.")

## 2. Helpers

In [ ]:
def _normalize_uri(uri):
    if not isinstance(uri, str): return uri
    return uri.strip().lower().replace('https://', 'http://')

def _normalize_path(p):
    if not isinstance(p, str): return p
    p = p.replace('\\', '/').replace('./', '').replace('.\/', '')
    if os.path.exists(p): return p
    fallback = os.path.join("results/trained_models", os.path.basename(p))
    return fallback if os.path.exists(fallback) else p

def _load_emb(model_name):
    path = f"results/embeddings/{model_name}_embeddings.csv"
    if not os.path.exists(path): raise FileNotFoundError(path)
    df = pd.read_csv(path, low_memory=False); col = f"{model_name}_embedding"
    
    def _parse(v):
        if pd.isna(v): return None
        try: return np.array(ast.literal_eval(str(v).replace('np.array(', '[').replace(')', ']')))
        except: return None
        
    df[col] = df[col].apply(_parse)
    
    def _is_valid(v):
        return isinstance(v, np.ndarray) and np.issubdtype(v.dtype, np.number) and not np.isnan(v).any()
        
    df = df[df[col].apply(_is_valid)].copy()
    if df.empty: return pd.DataFrame(), col
    df["instance"] = df["instance"].apply(_normalize_uri)
    
    # Select only instance and embedding for initial merge
    df = df[["instance", col]]
    
    cols_to_bring = ["instance"] + ALL_LABELS
    merged = df.merge(dataset[cols_to_bring], on="instance", how="inner")
    return merged, col

def build_sim_incremental(m, ids, exps_df, df, col):
    """Compute similarity hits for a list of experiment IDs."""
    X = np.stack(df[col].values)
    norm = np.linalg.norm(X, axis=1, keepdims=True); norm[norm == 0] = 1.0
    X_n = X / norm
    
    new_cols = {}
    for eid in ids:
        row = exps_df[exps_df['ID'] == eid].iloc[0]
        p = row['Parameters']; cl = DOMAIN_CLASSES_CORR[row['Domain']]
        th = 1.0 - p['distance']; mc = p.get('structure') == "2-multiclass"
        
        all_sims = np.full((len(X), len(cl)), -1.0)
        for i, cls_name in enumerate(cl):
            protos = p['prototypes'].get(cls_name)
            if protos is None: continue
            plist = list(protos.values()) if isinstance(protos, dict) else [protos]
            p_vecs = []
            for v in plist:
                if v is None: continue
                v_arr = np.array(v)
                if np.issubdtype(v_arr.dtype, np.number) and not np.isnan(v_arr).any():
                    p_vecs.append(v_arr)
            
            if not p_vecs: continue
            Pn = np.stack(p_vecs); pn_norm = np.linalg.norm(Pn, axis=1, keepdims=True); pn_norm[pn_norm == 0] = 1.0
            Pn = Pn / pn_norm
            all_sims[:, i] = (X_n @ Pn.T).max(axis=1)
        
        yp = np.zeros((len(X), len(cl)), dtype=int)
        if mc: yp[all_sims >= th] = 1
        else:
            best = np.argmax(all_sims, axis=1); mx = all_sims.max(axis=1)
            for ri, ci in enumerate(best): 
                if mx[ri] >= th: yp[ri, ci] = 1
        
        hits = (yp == df[cl].values).all(axis=1).astype(int)
        new_cols[eid] = hits
        print(f"      Computed {eid}: {np.mean(hits):.3f} accuracy")
    return pd.DataFrame(new_cols, index=df['instance'])

def build_clf_incremental(m, ids, exps_df, df, col):
    """Compute classifier hits for a list of experiment IDs."""
    X = np.stack(df[col].values)
    new_cols = {}
    for eid in ids:
        row = exps_df[exps_df['ID'] == eid].iloc[0]; p = row['Parameters']; cl = DOMAIN_CLASSES_CORR[row['Domain']]
        st = p.get("structure")
        pts = [_normalize_path(p.get("trained_classifier_hospital")), _normalize_path(p.get("trained_classifier_university_hospital"))] if st == "nested-class" else [_normalize_path(p.get("trained_classifier"))]
        if not all(pt and os.path.exists(pt) for pt in pts): continue
        try:
            if st == "nested-class":
                yh = load_trained_model(pts[0]).predict(X); yu = np.zeros_like(yh)
                pos = np.where(yh == 1)[0]
                if len(pos): yu[pos] = load_trained_model(pts[1]).predict(X[pos])
                hits = ((yh == df['hospital']) & (yu == df['university_hospital'])).astype(int)
            else:
                clf = load_trained_model(pts[0]); raw = clf.predict(X); yt = df[cl].values
                if yt.shape[1] > 1:
                    yp = np.zeros_like(yt)
                    if raw.ndim == 1: 
                        for i, v in enumerate(raw): 
                            if v < yt.shape[1]: yp[i, v] = 1
                    else: yp = raw.astype(int)
                else: yp = raw.reshape(-1, 1).astype(int)
                hits = (yp == yt).all(axis=1).astype(int)
            new_cols[eid] = hits
            print(f"      Computed {eid}: {np.mean(hits):.3f} accuracy")
        except Exception as e: print(f"Error {eid}: {e}")
    return pd.DataFrame(new_cols, index=df['instance'])

## 3. Build Tables (Incremental Caching)

In [ ]:
def _get_id(index, m, k, d):
    for eid in index.get(m, {}).get(k, []):
        if eid.startswith(d[:3]): return eid
    return None

def get_correctness_table(model_name, method_type, requested_ids, exps_df, build_fn):
    cache_path = os.path.join(CORRECTNESS_DIR, f"{model_name}_{method_type}.csv")
    cache_df = pd.DataFrame()
    if os.path.exists(cache_path):
        cache_df = pd.read_csv(cache_path, index_col=0)
        # Normalize index for safety
        cache_df.index = cache_df.index.map(_normalize_uri)
        
    missing_ids = [eid for eid in requested_ids if eid not in cache_df.columns]
    
    if not missing_ids:
        # print(f"    [CACHE] All {len(requested_ids)} experiments present for {model_name} {method_type}.")
        return cache_df[requested_ids]
        
    print(f"    [PROCESS] Computing {len(missing_ids)} missing experiments for {model_name} {method_type}...")
    df, col = _load_emb(model_name)
    if df.empty: return pd.DataFrame()
    
    new_results = build_fn(model_name, missing_ids, exps_df, df, col)
    
    if cache_df.empty: combined = new_results
    else: combined = cache_df.join(new_results, how="outer").fillna(0).astype(int)
    
    combined.to_csv(cache_path)
    # Return only the subset requested (might include both cached and fresh)
    return combined[requested_ids]

# 1. Initialize Indices
sim_index = {}; clf_index = {}
for _, r in sim_exps.iterrows(): 
    p = r['Parameters']
    if isinstance(p, dict): sim_index.setdefault(p.get('model'), {}).setdefault(p.get('n_shot'), []).append(r['ID'])
for _, r in clf_exps.iterrows(): 
    p = r['Parameters']
    if isinstance(p, dict):
        t = "lr" if "solver" in p else ("svm" if "kernel" in p else "unknown")
        clf_index.setdefault(p.get('model'), {}).setdefault(t, []).append(r['ID'])

# 2. Build Tables
sim_ct = {}; clf_ct = {}
for m in sim_index:
    if not m: continue
    ids = [i for s in sim_index[m].values() for i in s]
    ct = get_correctness_table(m, "sim", ids, sim_exps, build_sim_incremental)
    if not ct.empty: sim_ct[m] = ct

for m in clf_index:
    if not m: continue
    ids = [i for t in clf_index[m].values() for i in t]
    ct = get_correctness_table(m, "clf", ids, clf_exps, build_clf_incremental)
    if not ct.empty: clf_ct[m] = ct

print("\nAll tables ready.")

## 4. Tests & Plots

In [ ]:
all_res = []
def run_p_test(strata, lbl, tag):
    if not strata: return pd.DataFrame()
    r = stratified_permutation_test(strata, "A", "B", 10000, "pooled")
    return pd.DataFrame([{"analysis": tag, "comparison": lbl, "obs_diff": r[0], "p_value": r[1]}])

# A1: Shots
for sa, sb, lbl in [("1_shot", "0_shot", "1-shot vs 0-shot"), ("few_shot", "1_shot", "few vs 1-shot"), ("few_shot", "0_shot", "few vs 0-shot")]:
    st = []
    for m, ct in sim_ct.items():
        dfm = []
        for d in DOMAINS:
            ia, ib = _get_id(sim_index, m, sa, d), _get_id(sim_index, m, sb, d)
            if ia in ct.columns and ib in ct.columns: st.append(ct[[ia, ib]].rename(columns={ia: "A", ib: "B"}))
    all_res.append(run_p_test(st, lbl, "A1_shot"))

# A2: Sim Models
for sa in ["0_shot", "1_shot", "few_shot"]:
    mdls = sorted(sim_ct.keys())
    for ma, mb in combinations(mdls, 2):
        st = []
        for d in DOMAINS:
            ia, ib = _get_id(sim_index, ma, sa, d), _get_id(sim_index, mb, sa, d)
            if ma in sim_ct and mb in sim_ct and ia in sim_ct[ma].columns and ib in sim_ct[mb].columns:
                jo = sim_ct[ma][[ia]].join(sim_ct[mb][[ib]], how="inner").rename(columns={ia: "A", ib: "B"})
                if not jo.empty: st.append(jo)
        all_res.append(run_p_test(st, f"{ma} vs {mb}", f"A2_model_{sa}"))

# A3: LR vs SVM
st = []
for m, ct in clf_ct.items():
    lr = [_get_id(clf_index, m, "lr", d) for d in DOMAINS]; sv = [_get_id(clf_index, m, "svm", d) for d in DOMAINS]
    ia = [i for i in lr if i in ct.columns]; ib = [i for i in sv if i in ct.columns]
    if ia and ib: 
        st.append(pd.concat([ct[ia].astype(float).mean(axis=1), ct[ib].astype(float).mean(axis=1)], axis=1).rename(columns={0: "A", 1: "B"}))
all_res.append(run_p_test(st, "LR vs SVM", "A3_clf_head"))

# A4: Clf Models
for head in ["lr", "svm"]:
    mdls = sorted(clf_ct.keys())
    for ma, mb in combinations(mdls, 2):
        st = []
        for d in DOMAINS:
            ia, ib = _get_id(clf_index, ma, head, d), _get_id(clf_index, mb, head, d)
            if ma in clf_ct and mb in clf_ct and ia in clf_ct[ma].columns and ib in clf_ct[mb].columns:
                jo = clf_ct[ma][[ia]].join(clf_ct[mb][[ib]], how="inner").rename(columns={ia: "A", ib: "B"})
                if not jo.empty: st.append(jo)
        all_res.append(run_p_test(st, f"{ma} vs {mb}", f"A4_model_{head}"))

## 5. Summary

In [ ]:
final = pd.concat([r for r in all_res if not r.empty], ignore_index=True)
if not final.empty:
    rej, pc, _, _ = multipletests(final['p_value'], method="holm")
    final["p_corrected"] = pc; final["significant"] = rej
    final.to_csv("results/embedding_permutation_tests.csv", index=False)

    sns.set_theme(style="whitegrid", palette="pastel")
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    for i, t in enumerate(["A1_shot", "A2_model", "A3_clf_head", "A4_model"]):
        sub = final[final['analysis'].str.startswith(t)]
        if sub.empty: continue
        clrs = ["#2ecc71" if (s and d > 0) else ("#e74c3c" if s else "#95a5a6") for s, d in zip(sub['significant'], sub['obs_diff'])]
        sns.barplot(data=sub, x="comparison", y="obs_diff", ax=axes[i], palette=clrs)
        axes[i].set_title(t, fontweight='bold'); axes[i].tick_params(axis='x', rotation=45)
        for p, bar in zip(sub['p_corrected'], axes[i].patches):
            if p < 0.05: axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height(), '*', ha='center')
    plt.tight_layout(); fig.savefig("figures/perm_tests.png", dpi=300); plt.show()
    display(final)
else:
    print("No results to display.")